In [21]:
from minio import Minio
import os
from pathlib import Path
from __future__ import annotations
from dataclasses import dataclass
from functools import lru_cache

In [17]:
from dotenv import load_dotenv
from minio import Minio
load_dotenv("/home/rhadamanthys/Data-Handler/.env")

True

In [27]:
def _env_bool(value: str | None, default: bool = False) -> bool:
    if value is None:
        return default
    return value.lower() in {"1", "true", "yes", "on"}

In [28]:
@dataclass(frozen=True)
class MinioSettings:
    endpoint: str
    access_key: str
    secret_key: str
    region: str | None = None
    secure: bool = False

In [29]:
def get_minio_settings() -> MinioSettings:
    print(os.getenv("MINIO_ENDPOINT"))
    endpoint = os.getenv("MINIO_ENDPOINT")
    access_key = os.getenv("MINIO_ROOT_USER")
    secret_key = os.getenv("MINIO_ROOT_PASSWORD")
    region = os.getenv("MINIO_REGION")
    secure = _env_bool(os.getenv("MINIO_USE_SSL"), default=False)
    if not endpoint or not access_key or not secret_key:
        raise RuntimeError("MINIO_ENDPOINT, MINIO_ACCESS_KEY, and MINIO_SECRET_KEY must be set")
    return MinioSettings(endpoint, access_key, secret_key,secure)

In [30]:
@lru_cache()
def get_minio_client() -> Minio:
    settings = get_minio_settings()
    return Minio(
        settings.endpoint,
        access_key=settings.access_key,
        secret_key=settings.secret_key,
        secure=settings.secure,
    )

In [41]:
def ensure_bucket(bucket_name: str) -> None:
    client = get_minio_client()
    try:
        if client.bucket_exists(bucket_name):
            return
        client.make_bucket(bucket_name)
        logger.info("Created MinIO bucket %s", bucket_name)
    except S3Error as exc:
        logger.exception("Could not ensure bucket %s: %s", bucket_name, exc)
        raise BucketCreationError(str(exc)) from exc

In [46]:
def upload_file(bucket: str, source_path: Path, object_name: str, content_type: str | None = None) -> None:
    client = get_minio_client()
    ensure_bucket(bucket)
    try:
        client.fput_object(bucket, object_name, str(source_path), content_type=content_type)
    except Exeption as exc:
        raise ObjectUploadError(str(exc)) from exc

In [47]:
abs_paths = []
for root, dirs, files in os.walk("/home/rhadamanthys/Data-Handler/data/tmp"):
    for file in files:
        abs_path = os.path.abspath(os.path.join(root, file))
        if "github" in abs_path:
            abs_paths.append(abs_path)

In [48]:
abs_paths

['/home/rhadamanthys/Data-Handler/data/tmp/github::ECTSum/prepare_data_ectbps_ext.py',
 '/home/rhadamanthys/Data-Handler/data/tmp/github::ECTSum/prepare_data_ectbps_para_mask.py',
 '/home/rhadamanthys/Data-Handler/data/tmp/github::ECTSum/utils.py',
 '/home/rhadamanthys/Data-Handler/data/tmp/github::ECTSum/prepare_data_ectbps_para.py',
 '/home/rhadamanthys/Data-Handler/data/tmp/github::ECTSum/evaluate_with_mask.py',
 '/home/rhadamanthys/Data-Handler/data/tmp/github::ECTSum/LICENSE.txt',
 '/home/rhadamanthys/Data-Handler/data/tmp/github::ECTSum/README.md',
 '/home/rhadamanthys/Data-Handler/data/tmp/github::ECTSum/evaluate.py',
 '/home/rhadamanthys/Data-Handler/data/tmp/github::ECTSum/data/final/train/ects/ARW_q2_2020.txt',
 '/home/rhadamanthys/Data-Handler/data/tmp/github::ECTSum/data/final/train/ects/MSM_q4_2021.txt',
 '/home/rhadamanthys/Data-Handler/data/tmp/github::ECTSum/data/final/train/ects/TDC_q2_2021.txt',
 '/home/rhadamanthys/Data-Handler/data/tmp/github::ECTSum/data/final/trai

In [50]:
for file_path in abs_paths:
    key = file_path.replace("/home/rhadamanthys/Data-Handler/data/tmp/","")
    dataset_id = [ i for i in file_path.split("/") if "github::" in i][0]
    file_path = Path(file_path).resolve()
    workspace = Path("/home/rhadamanthys/Data-Handler/data/tmp").resolve()
    relative = file_path.relative_to(workspace)
    print(relative)
    upload_file("github-raw", file_path, key)


github::ECTSum/prepare_data_ectbps_ext.py
github::ECTSum/prepare_data_ectbps_para_mask.py
github::ECTSum/utils.py
github::ECTSum/prepare_data_ectbps_para.py
github::ECTSum/evaluate_with_mask.py
github::ECTSum/LICENSE.txt
github::ECTSum/README.md
github::ECTSum/evaluate.py
github::ECTSum/data/final/train/ects/ARW_q2_2020.txt
github::ECTSum/data/final/train/ects/MSM_q4_2021.txt
github::ECTSum/data/final/train/ects/TDC_q2_2021.txt
github::ECTSum/data/final/train/ects/OI_q2_2021.txt
github::ECTSum/data/final/train/ects/UNF_q1_2021.txt
github::ECTSum/data/final/train/ects/AKR_q4_2020.txt
github::ECTSum/data/final/train/ects/ITT_q3_2021.txt
github::ECTSum/data/final/train/ects/NP_q3_2021.txt
github::ECTSum/data/final/train/ects/SYY_q1_2022.txt
github::ECTSum/data/final/train/ects/CPT_q3_2020.txt
github::ECTSum/data/final/train/ects/ELY_q4_2020.txt
github::ECTSum/data/final/train/ects/BLX_q4_2018.txt
github::ECTSum/data/final/train/ects/WHG_q2_2020.txt
github::ECTSum/data/final/train/ects/FR_

NameError: name 'Exeption' is not defined